In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score, classification_report, confusion_matrix
from scipy.stats import randint as sp_randint

import shap
import joblib

In [ ]:
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')

# Data Loading and Inspection

In [ ]:
df_gym= pd.read_csv('data/1. Gym Members Exercise Dataset/gym_members_exercise_tracking.csv')
df_fit = pd.read_csv('data/2. Exercise and Fitness Metrics Dataset/exercise_dataset.csv')
df_excise= pd.read_csv('data/3. Fitness Exercises Dataset/exercises_flat.csv')
df_rec = pd.read_excel('data/4. Mendeley Gym Recommendation Dataset/gym recommendation.xlsx')

print("Columns in gym_members_exercise_tracking.csv:", df_gym.columns.tolist())
print("Columns in exercise_dataset.csv:", df_fit.columns.tolist())
print("Columns in exercises_flat.csv:", df_excise.columns.tolist())
print("Columns in gym recommendation.xlsx:", df_rec.columns.tolist())

print()

print(f'Dataset 1 (Gym Members)          : {df_gym.shape[0]:,} rows × {df_gym.shape[1]} cols')
print(f'Dataset 2 (Exercise & Fitness)   : {df_fit.shape[0]:,} rows × {df_fit.shape[1]} cols')
print(f'Dataset 3 (Exercise Library)     : {df_excise.shape[0]:,} rows × {df_excise.shape[1]} cols')
print(f'Dataset 4 (Mendeley Coaching)    : {df_rec.shape[0]:,} rows × {df_rec.shape[1]} cols')
print()

In [ ]:
print(df_gym.head(3).to_string())


In [ ]:
print(df_fit.head(3).to_string())

In [ ]:
print(df_excise.head(3).to_string())

In [ ]:
print(df_rec.head(3).to_string())

In [ ]:
for label, df in [("D1 Gym Members", df_gym), ("D2 Fitness", df_fit),("D3 Exercises", df_excise), ("D4 Recommendation", df_rec)]:
    missing = df.isnull().sum()
    has_missing = missing[missing > 0]
    if len(has_missing):
        print(f"\n{label}:\n{has_missing}")
    else:
        print(f"{label}: No missing values")

In [ ]:
print("\nDataset 1 (Gym Members) Descriptive Stats")
print(df_gym.describe().round(2).to_string())
 
print("\nDataset 2 (Fitness) Descriptive Stats")
print(df_fit.describe().round(2).to_string())

print("\nDataset 3 (Exercises) Descriptive Stats")
print(df_excise.describe().round(2).to_string())

print("\nDataset 4 (Recommendation) Descriptive Stats")
print(df_rec.describe().round(2).to_string())

In [ ]:
print("Dataset 1 (Gym Members) Info:")
df_gym.info()

print("\nDataset 2 (Fitness) Info:")
df_fit.info()

print("\nDataset 3 (Exercises) Info:")
df_excise.info()

print("\nDataset 4 (Recommendation) Info:")
df_rec.info()

In [ ]:
df_gym.head(3)

In [ ]:
df_gym.head(3)

In [ ]:
df_excise.head(3)

In [ ]:
df_rec.head(3)

# Data Preprocessing

In [ ]:
df_fit_clean = df_fit.copy()

df_fit_clean.drop(columns=['ID', 'Exercise'], inplace=True)

df_fit_clean.rename(columns={
    'Calories Burn' : 'Calories_Burned',
    'Actual Weight' : 'Weight (kg)',
    'Dream Weight' : 'Dream_Weight',
    'Heart Rate' : 'Avg_BPM',
    'Weather Conditions' : 'Weather_Conditions',
    'Exercise Intensity' : 'Exercise_Intensity',
}, inplace=True)

df_fit_clean.rename(columns={'Duration': 'Session_Duration (hours)'}, inplace=True)
df_fit_clean['Session_Duration (hours)'] = df_fit_clean['Session_Duration (hours)'] / 60

#Adding new column name _source to dataset 1 and 2 to identify the source of data
df_gym['_source']      = 'ds1'
df_fit_clean['_source'] = 'ds2'

In [ ]:
print('DS1 columns:', list(df_gym.columns))
print()
print('DS2 columns After CLeaning and Renaming:', list(df_fit_clean.columns))

In [ ]:
# pd.concat fills missing columns with NaN automatically
dfUnified = pd.concat([df_gym, df_fit_clean], ignore_index=True)

print(f'Merged DataFrame shape: {dfUnified.shape}')
print(f'  DS1 (gym data) rows: {(dfUnified["_source"]=="ds1").sum()}')
print(f'  DS2 (fit data) rows: {(dfUnified["_source"]=="ds2").sum()}')
print()
print('Missing values per column:')
print(dfUnified.isnull().sum().to_string())

In [ ]:
dfUnified.head(3)

In [ ]:
#Training Random Forest Regressor for each column using dataset one then predict realistic values for dataset two

#First : DS1 only columns (train on DS1, predict for DS2)
IMPUTE_PREDICTORS = [
    'Age', 'Weight (kg)', 'BMI',
    'Avg_BPM', 'Session_Duration (hours)'
]

# Encode Gender temporarily just for imputation models
dfUnified['_g'] = (dfUnified['Gender'] == 'Male').astype(int)
imp_preds = IMPUTE_PREDICTORS + ['_g']

ds1_mask = dfUnified['_source'] == 'ds1'
ds2_mask = dfUnified['_source'] == 'ds2'

imputer_models = {} # save each model (needed to impute inference time data)
impute_report  = []

DS1_ONLY_COLS = [
    'Height (m)',
    'Fat_Percentage', 'Water_Intake (liters)',
    'Workout_Frequency (days/week)', 'Experience_Level'
]

dfUnified.loc[ds2_mask, 'Max_BPM'] = 220 - dfUnified.loc[ds2_mask, 'Age']

# Impute Resting_BPM for DS2 using RF trained on DS1 (same pattern as other DS1-only cols)
_resting_preds = IMPUTE_PREDICTORS + ['_g']
_X_known_r = dfUnified.loc[ds1_mask, _resting_preds]
_y_known_r = dfUnified.loc[ds1_mask, 'Resting_BPM']
_X_miss_r  = dfUnified.loc[ds2_mask, _resting_preds]

_imp_resting = RandomForestRegressor(
    n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
)
_imp_resting.fit(_X_known_r, _y_known_r)
_cv_r2_resting = cross_val_score(
    _imp_resting, _X_known_r, _y_known_r, cv=5, scoring='r2', n_jobs=-1
).mean()
dfUnified.loc[ds2_mask, 'Resting_BPM'] = _imp_resting.predict(_X_miss_r)
imputer_models['Resting_BPM'] = _imp_resting
print(f'  Resting_BPM (DS1→DS2)  CV R²={_cv_r2_resting:.3f}')

print('DS1-only columns (train on DS1, predict for DS2):\n')
for col in DS1_ONLY_COLS:
    missing_n = dfUnified[col].isnull().sum()
    if missing_n == 0:
        print(f'  {col:<42s} — no NaNs, skipped')
        continue

    X_known  = dfUnified.loc[ds1_mask, imp_preds]
    y_known  = dfUnified.loc[ds1_mask, col]
    X_miss   = dfUnified.loc[ds2_mask, imp_preds]

    imp_rf = RandomForestRegressor(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    )
    imp_rf.fit(X_known, y_known)
    train_r2 = imp_rf.score(X_known, y_known)
    cv_r2    = cross_val_score(
        imp_rf, X_known, y_known, cv=5, scoring='r2', n_jobs=-1
    ).mean()

    dfUnified.loc[ds2_mask, col] = imp_rf.predict(X_miss)
    imputer_models[col] = imp_rf
    impute_report.append({
        'Column': col, 'Direction': 'DS1→DS2',
        'Missing': missing_n,
        'Train R²': round(train_r2, 3),
        'CV R² (5-fold)': round(cv_r2, 3)
    })
    print(f'  {col:<42s} Train R²={train_r2:.3f}  CV R²={cv_r2:.3f}')

DS2_ONLY_COLS = ['Exercise_Intensity', 'Dream_Weight']

#Second : DS2 only columns (train on DS2, predict for DS1)
print('\nDS2-only columns (train on DS2, predict for DS1):\n')
for col in DS2_ONLY_COLS:
    missing_n = dfUnified[col].isnull().sum()
    if missing_n == 0:
        print(f'  {col:<42s} — no NaNs, skipped')
        continue

    X_known  = dfUnified.loc[ds2_mask, imp_preds]
    y_known  = dfUnified.loc[ds2_mask, col]
    X_miss   = dfUnified.loc[ds1_mask, imp_preds]

    imp_rf = RandomForestRegressor(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    )
    imp_rf.fit(X_known, y_known)
    train_r2 = imp_rf.score(X_known, y_known)
    cv_r2    = cross_val_score(
        imp_rf, X_known, y_known, cv=5, scoring='r2', n_jobs=-1
    ).mean()

    dfUnified.loc[ds1_mask, col] = imp_rf.predict(X_miss)
    imputer_models[col] = imp_rf
    impute_report.append({
        'Column': col, 'Direction': 'DS2→DS1',
        'Missing': missing_n,
        'Train R²': round(train_r2, 3),
        'CV R² (5-fold)': round(cv_r2, 3)
    })
    print(f'  {col:<42s} Train R²={train_r2:.3f}  CV R²={cv_r2:.3f}')

dfUnified.drop(columns=['_g'], inplace=True)

print()
print(pd.DataFrame(impute_report).to_string(index=False))
remaining = dfUnified.select_dtypes(include='number').isnull().sum().sum()
print(f'\nTotal remaining numeric NaNs: {remaining}')

In [ ]:
# Train a Random Forest classifier on the rows that HAVE the value,predict for rows that don't (predicting Gender for DS2 using DS1 data)

cat_report = []

def impute_categorical(col, train_mask, predict_mask, predictors):
    dfUnified['_g2'] = (dfUnified['Gender'] == 'Male').astype(int)
    feats = [p for p in predictors + ['_g2'] if p in dfUnified.columns]

    X_known  = dfUnified.loc[train_mask,   feats]
    y_known  = dfUnified.loc[train_mask,   col]
    X_miss   = dfUnified.loc[predict_mask, feats]

    clf = RandomForestClassifier(
        n_estimators=200, max_depth=8,
        random_state=42, n_jobs=-1
    )

    clf.fit(X_known, y_known)
    cv_acc = cross_val_score(
        clf, X_known, y_known,
        cv=5, scoring='accuracy', n_jobs=-1
    ).mean()
    dfUnified.loc[predict_mask, col] = clf.predict(X_miss)
    dfUnified.drop(columns=['_g2'], inplace=True)
    return clf, round(cv_acc, 3)

base_preds = [
    'Age', 'Weight (kg)', 'BMI',
    'Avg_BPM', 'Session_Duration (hours)'
]

# Workout_Type: train on DS1, predict for DS2
if dfUnified['Workout_Type'].isnull().sum() > 0:
    clf_wt, acc_wt = impute_categorical(
        'Workout_Type', ds1_mask, ds2_mask, base_preds
    )
    print(f'  Workout_Type  CV Accuracy = {acc_wt:.3f}')
    cat_report.append({'Column': 'Workout_Type', 'CV Accuracy': acc_wt})

# Weather_Conditions: train on DS2, predict for DS1
if dfUnified['Weather_Conditions'].isnull().sum() > 0:
    clf_wc, acc_wc = impute_categorical(
        'Weather_Conditions', ds2_mask, ds1_mask,
        base_preds + ['Exercise_Intensity']
    )
    print(f'  Weather_Conditions    CV Accuracy = {acc_wc:.3f}')
    cat_report.append({'Column': 'Weather_Conditions', 'CV Accuracy': acc_wc})

print()
print(pd.DataFrame(cat_report).to_string(index=False))
print(f'\nRemaining categorical NaNs: \n{dfUnified[["Workout_Type","Weather_Conditions"]].isnull().sum().to_string()}')

In [ ]:
dfUnified['dataset_source'] = (dfUnified['_source'] == 'ds2').astype(int) # 0=DS1 (gym), 1=DS2 (fitness)
print('dataset_source added: 0=DS1 (gym), 1=DS2 (fitness)')
print(dfUnified.groupby('dataset_source')['Calories_Burned'].describe().round(1))

dfUnified.drop(columns=['_source'], inplace=True)
print('\nRemaining NaNs:', dfUnified.isnull().sum().sum())

In [ ]:
beforeLength = len(dfUnified)
dfUnified.drop_duplicates(inplace=True)
print('Duplicates removed:', beforeLength - len(dfUnified))

In [ ]:
Q1 = dfUnified['Calories_Burned'].quantile(0.25)
Q3 = dfUnified['Calories_Burned'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR

In [ ]:
afterLength = len(dfUnified)
dfUnified = dfUnified[(dfUnified['Calories_Burned'] >= lower) & (dfUnified['Calories_Burned'] <= upper)]
print(f'Outliers removed: {afterLength - len(dfUnified)}')

In [ ]:
print('Final Shape:', dfUnified.shape)

In [ ]:
dfUnified['Gender'] = dfUnified['Gender'].map({'Male': 1, 'Female': 0}) #Male: 1, Female: 0

dfUnified = pd.get_dummies(dfUnified, columns=['Workout_Type'], prefix='wt', drop_first=False) #encoding for Workout_Type: wt_Cardio, wt_Strength, wt_Flexibility

weather_map = {'Sunny': 2, 'Cloudy': 1, 'Rainy': 0}
dfUnified['Weather_Conditions'] = dfUnified['Weather_Conditions'].map(weather_map)
# Safety net only — impute_categorical() should have filled all Weather_Conditions NaNs
_wc_remaining = dfUnified['Weather_Conditions'].isnull().sum()
if _wc_remaining > 0:
    print(f'Warning: {_wc_remaining} Weather_Conditions NaNs remain after imputation — filling with median')
    dfUnified['Weather_Conditions'] = dfUnified['Weather_Conditions'].fillna(
        dfUnified['Weather_Conditions'].median()
    ) #Sunny=2, Cloudy=1, Rainy=0

bool_cols = dfUnified.select_dtypes(include='bool').columns
dfUnified[bool_cols] = dfUnified[bool_cols].astype(int) #convert bool into int (True=1, False=0) for wt_Cardio, wt_Strength, wt_Flexibility

In [ ]:
print('Encoded columns:', [c for c in dfUnified.columns if c.startswith('wt_') or c in ['Gender','Weather_Conditions']])
print(f'DataFrame shape after encoding: {dfUnified.shape}')
dfUnified.head(3)

In [ ]:
dfUnified.columns.tolist()

In [ ]:
dfUnified.dtypes

# Feature Engineering

In [ ]:
# ── Feature Engineering ─────────────────────────────────────────────────
# Creating features from columns available at PREDICTION TIME.
# At runtime the app has: Age, Gender, Weight, Height, BMI (auto),
# Session_Duration, Avg_BPM, Exercise_Intensity, Weather_Conditions,
# Workout_Type, Water_Intake, Workout_Frequency, Resting_BPM, Fat_Percentage

dfUnified['Max_BPM_calc'] = 220 - dfUnified['Age']

# Heart Rate Reserve = Max - Resting BPM (Karvonen method)
dfUnified['HR_Reserve'] = dfUnified['Max_BPM_calc'] - dfUnified['Resting_BPM']
# HR × Duration: primary interaction (exertion × time)
dfUnified['HR_Duration']        = dfUnified['Avg_BPM'] * dfUnified['Session_Duration (hours)']
# Intensity × Duration
dfUnified['Intensity_Duration'] = dfUnified['Exercise_Intensity'] * dfUnified['Session_Duration (hours)']
# Weight × Duration: mass × effort
dfUnified['Weight_Duration']    = dfUnified['Weight (kg)'] * dfUnified['Session_Duration (hours)']
# BMI × Duration
dfUnified['BMI_Duration']       = dfUnified['BMI'] * dfUnified['Session_Duration (hours)']
# HR Intensity Ratio: % of max HR — indicates effort zone
dfUnified['HR_Intensity_Ratio'] = dfUnified['Avg_BPM'] / dfUnified['Max_BPM_calc']
dfUnified['log_Calories'] = np.log1p(dfUnified['Calories_Burned'])
print('Engineered features added.')
print('Features:', ['Max_BPM_calc','HR_Reserve','HR_Duration','Intensity_Duration',
                    'Weight_Duration','BMI_Duration','HR_Intensity_Ratio','log_Calories'])
print(f'DataFrame shape: {dfUnified.shape}')

# EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(dfUnified['Calories_Burned'], bins=50, color='darkturquoise', edgecolor='white')
axes[0].set_title('Distribution of Calories Burned')
axes[0].set_xlabel('Calories Burned')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(dfUnified['Calories_Burned'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='darkturquoise', alpha=0.6))
axes[1].set_title('Calories Burned Boxplot')
axes[1].set_ylabel('Calories Burned')

plt.tight_layout()
plt.show()

In [ ]:
numeric_Df = dfUnified.select_dtypes(include='number')
corr = numeric_Df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='mako',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
scatter_features = [
    'Session_Duration (hours)', 'Avg_BPM', 'Weight (kg)', 'BMI', 'Age'
]

fig, axes = plt.subplots(1, len(scatter_features), figsize=(18, 4))
for ax, feat in zip(axes, scatter_features):
    ax.scatter(dfUnified[feat], dfUnified['Calories_Burned'],
               alpha=0.2, s=10, color='darkturquoise')
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('Calories Burned', fontsize=9)
    ax.set_title(feat, fontsize=9)

plt.suptitle('Feature vs Calories Burned', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
hist_features = ['Age', 'Weight (kg)', 'Height (m)', 'BMI',
                 'Session_Duration (hours)', 'Workout_Frequency (days/week)', 'Water_Intake (liters)', 'Fat_Percentage']
hist_features = [f for f in hist_features if f in dfUnified.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, feat in enumerate(hist_features):
    axes[i].hist(dfUnified[feat].dropna(), bins=30,
                 color=sns.color_palette('muted')[i % 8], edgecolor='white')
    axes[i].set_title(feat, fontsize=9)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gender_means = dfUnified.groupby('Gender')['Calories_Burned'].mean()
axes[0].bar(['Female (0)', 'Male (1)'], gender_means.values,
            color=['pink','teal'])
axes[0].set_title('Average Calories Burned by Gender')
axes[0].set_ylabel('Avg Calories')

wt_cols = [c for c in dfUnified.columns if c.startswith('wt_')]
if wt_cols:
    dfUnified['_wt_label'] = dfUnified[wt_cols].idxmax(axis=1).str.replace('wt_', '')
    wt_means = dfUnified.groupby('_wt_label')['Calories_Burned'].mean().sort_values()
    wt_means.plot(kind='barh', ax=axes[1], color='darkturquoise', edgecolor='white')
    axes[1].set_title('Average Calories Burned by Workout Type')
    axes[1].set_xlabel('Avg Calories')
    dfUnified.drop(columns=['_wt_label'], inplace=True)

plt.tight_layout()
plt.show()

# Feature Selection

In [ ]:
# ── Multi-Method Feature Selection ──────────────────────────────────────
# BUG 2 FIX: dataset_source was wrongly excluded.
# DS1 (gym) = 200-1800 cal, DS2 (fitness) = 100-500 cal.
# Without this flag the model is blind to which distribution to predict.
# At inference time the web app sets dataset_source=0 (gym session).
EXCLUDE = {'Calories_Burned', 'log_Calories', 'Dream_Weight', 'Max_BPM'}
ALL_FEATURES = [c for c in dfUnified.columns if c not in EXCLUDE]

X_fs = dfUnified[ALL_FEATURES].copy()
y_fs = dfUnified['Calories_Burned']

print(f'Feature pool ({len(ALL_FEATURES)}): {ALL_FEATURES}\n')

print('Method 1 — RF Importance...')
rf_fs = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_fs.fit(X_fs, y_fs)
rf_imp = pd.Series(rf_fs.feature_importances_, index=ALL_FEATURES)

print('Method 2 — Permutation Importance...')
perm = permutation_importance(
    rf_fs, X_fs, y_fs, n_repeats=15, random_state=42, n_jobs=-1
)
perm_imp = pd.Series(np.clip(perm.importances_mean, 0, None), index=ALL_FEATURES)

print('Method 3 — Mutual Information...')
assert X_fs.isnull().sum().sum() == 0, 'NaNs still present in X_fs!'
mi = mutual_info_regression(X_fs, y_fs, random_state=42)
mi_imp = pd.Series(mi, index=ALL_FEATURES)

print('Method 4 — SHAP values...')
shap_sample = X_fs.sample(min(800, len(X_fs)), random_state=42)
explainer   = shap.TreeExplainer(rf_fs)
shap_vals   = explainer.shap_values(shap_sample)
shap_imp    = pd.Series(np.abs(shap_vals).mean(axis=0), index=ALL_FEATURES)

def norm01(s):
    rng = s.max() - s.min()
    return (s - s.min()) / (rng + 1e-12)

fs_scores = pd.DataFrame({
    'RF Importance'         : norm01(rf_imp),
    'Permutation Importance': norm01(perm_imp),
    'Mutual Information'    : norm01(mi_imp),
    'SHAP'                  : norm01(shap_imp),
})
fs_scores['Combined Score'] = fs_scores.mean(axis=1)
fs_scores = fs_scores.sort_values('Combined Score', ascending=False)
print('\n' + fs_scores.round(3).to_string())

In [ ]:
THRESHOLD = 0.05
selected_features = fs_scores[fs_scores['Combined Score'] > THRESHOLD].index.tolist()

print(f'\n{len(selected_features)} features selected (combined score > {THRESHOLD}):')
print(selected_features)

In [ ]:
top_n = min(15, len(fs_scores))
top_feats = fs_scores.head(top_n)
methods = ['RF Importance', 'Permutation Importance', 'Mutual Information', 'SHAP']
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
colors = ['lightblue', 'teal', 'pink', 'lightgreen']
for ax, method, color in zip(axes, methods, colors):
    ax.barh(top_feats.index[::-1], top_feats[method][::-1], color=color)
    ax.set_title(method, fontsize=10)
    ax.set_xlabel('Normalized Score', fontsize=8)
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle('Feature Importance 4 Independent Methods', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
palette = ['navy' if s > THRESHOLD else 'lightgreen'
           for s in fs_scores['Combined Score']]
plt.barh(fs_scores.index[::-1], fs_scores['Combined Score'][::-1], color=palette[::-1])
plt.axvline(THRESHOLD, color='red', linestyle='--', linewidth=1, label=f'Threshold {THRESHOLD}')
plt.title('Combined Feature Score (avg of 4 methods)')
plt.xlabel('Combined Score')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
shap.summary_plot(
    shap_vals, shap_sample,
    feature_names=ALL_FEATURES,
    max_display=15, show=True
)

In [ ]:
# ── Train / Test split ───────────────────────────────────────────────────
# Double-check: log_Calories and Calories_Burned must NOT be in features
assert 'log_Calories'   not in selected_features, 'DATA LEAKAGE: log_Calories in features!'
assert 'Calories_Burned' not in selected_features, 'DATA LEAKAGE: Calories_Burned in features!'
assert 'Dream_Weight'   not in selected_features, 'Dream_Weight should not be in features!'
print('Leakage check passed.')

X     = dfUnified[selected_features].copy()
y_raw = dfUnified['Calories_Burned'].copy()  # original scale (eval only)
y     = dfUnified['log_Calories'].copy()      # log scale (training)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
_idx       = y_test.index
y_test_raw = y_raw.loc[_idx]

# Keep source labels for per-source diagnostics only (NOT a feature)
src_test = dfUnified.loc[_idx, 'dataset_source'] if 'dataset_source' in dfUnified.columns else None

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Features: {selected_features}')
print(f'Training target (log): [{y_train.min():.2f}, {y_train.max():.2f}]')
print(f'Test target (cal):     [{y_test_raw.min():.0f}, {y_test_raw.max():.0f}]')

In [ ]:
ds_train = dfUnified.loc[X_train.index, 'dataset_source']
w_ds1 = len(ds_train) / (2 * (ds_train == 0).sum())  # upweight DS1
w_ds2 = len(ds_train) / (2 * (ds_train == 1).sum())  # downweight DS2
sample_weights = ds_train.map({0: w_ds1, 1: w_ds2}).values
print(f'Sample weight DS1: {w_ds1:.3f}  DS2: {w_ds2:.3f}')

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Scaling done. Train mean:', X_train_sc[:, :3].mean(axis=0).round(3))

# Training Model

In [ ]:
CALORIE_BINS   = [0, 400, 700, 1000, 9999]
CALORIE_LABELS = ['Low', 'Medium', 'High', 'Very High']
results = {}

from sklearn.metrics import make_scorer

def _r2_original_scale(estimator, X, y_log):
    """Custom scorer: predicts in log space, scores in original calorie scale.
    BUG 3 FIX: CV R² is now in original space (same as test R²), making
    them directly comparable. Previously CV R² was in log space while
    test R² was in original space — they measured different things.
    """
    pred_log = estimator.predict(X)
    pred_cal = np.expm1(np.clip(pred_log, 0, None))
    y_cal    = np.expm1(np.clip(y_log, 0, None))
    return r2_score(y_cal, pred_cal)

_orig_scorer = make_scorer(_r2_original_scale)

def evaluate(name, model, X_tr, X_te, y_tr, y_te_log, y_te_raw,
             use_scaled=False, sample_w=None):
    Xtr = X_tr if not use_scaled else X_train_sc
    Xte = X_te if not use_scaled else X_test_sc

    pred_log = model.predict(Xte)
    pred     = np.expm1(np.clip(pred_log, 0, None))
    actual   = y_te_raw.values

    mae  = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2   = r2_score(actual, pred)

    mask = actual > 0
    mape = np.mean(np.abs((actual[mask]-pred[mask])/actual[mask])) * 100

    errors  = np.abs(actual - pred)
    acc_50  = (errors <=  50).mean() * 100
    acc_100 = (errors <= 100).mean() * 100
    acc_150 = (errors <= 150).mean() * 100

    ab = pd.cut(pd.Series(actual), bins=CALORIE_BINS,
                labels=CALORIE_LABELS).astype(str)
    pb = pd.cut(pd.Series(pred),   bins=CALORIE_BINS,
                labels=CALORIE_LABELS).astype(str)
    f1_w = f1_score(ab, pb, average='weighted', zero_division=0)

    # BUG 4 FIX: pass sample_weight to CV fit via fit_params
    fit_params = {'sample_weight': sample_w} if sample_w is not None else {}
    cv_s = cross_val_score(model, Xtr, y_tr, cv=5,
                           scoring=_orig_scorer,
                           fit_params=fit_params,
                           n_jobs=-1)

    results[name] = {
        'MAE':mae,'RMSE':rmse,'R2':r2,'MAPE':mape,
        'Acc@50':acc_50,'Acc@100':acc_100,'Acc@150':acc_150,
        'BinnedF1':f1_w,
        'CV_R2_mean':cv_s.mean(),'CV_R2_std':cv_s.std()
    }
    print(f'  MAE        = {mae:.2f} cal')
    print(f'  RMSE       = {rmse:.2f} cal')
    print(f'  R²         = {r2:.4f}')
    print(f'  MAPE       = {mape:.2f}%')
    print(f'  Acc ±50 cal = {acc_50:.1f}%')
    print(f'  Acc ±100 cal= {acc_100:.1f}%')
    print(f'  Acc ±150 cal= {acc_150:.1f}%')
    print(f'  Binned F1  = {f1_w:.4f}')
    print(f'  CV R² (orig) = {cv_s.mean():.4f} ± {cv_s.std():.4f}')
    return pred

print('evaluate() ready. CV R² now in original calorie scale (comparable with test R²).')

In [ ]:
#print('Linear Regression:')
#lr = LinearRegression()
#lr.fit(X_train_sc, y_train)
#pred_lr = evaluate('Linear Regression', lr,
#                   X_train_sc, X_test_sc, y_train, y_test,
#                   use_scaled=True)

In [ ]:
print('Ridge Regression + Polynomial Interactions:')
lr_pipe = Pipeline([
    ('poly',  PolynomialFeatures(degree=2, interaction_only=True,
                                 include_bias=False)),
    ('ridge', Ridge(alpha=10.0))
])
lr_pipe.fit(X_train_sc, y_train)
# Ridge doesn't support sample_weight in Pipeline CV cleanly; omit for LR
pred_lr = evaluate('Linear Regression', lr_pipe,
                   X_train_sc, X_test_sc, y_train, y_test, y_test_raw,
                   use_scaled=True)
lr = lr_pipe

In [ ]:
print('Random Forest Regressor:')
rf = RandomForestRegressor(
    n_estimators=300, max_depth=15,
    min_samples_split=3, min_samples_leaf=2,
    max_features='sqrt', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train, sample_weight=sample_weights)
pred_rf = evaluate('Random Forest', rf,
                   X_train, X_test, y_train, y_test, y_test_raw,
                   sample_w=sample_weights)

In [ ]:
print('XGBoost Regressor:')
# BUG 5 FIX: eval_set previously used y_test (test-set leakage).
# Now uses a proper validation split from training data.
_Xt2, _Xv, _yt2, _yv, _sw2, _swv = train_test_split(
    X_train, y_train, sample_weights, test_size=0.15, random_state=42
)
xgb = XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=4,
    subsample=0.9, colsample_bytree=0.9,
    reg_alpha=0.1, reg_lambda=1,
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42, verbosity=0
)
xgb.fit(_Xt2, _yt2,
        sample_weight=_sw2,
        eval_set=[(_Xv, _yv)],
        verbose=False)
pred_xgb = evaluate('XGBoost', xgb,
                    X_train, X_test, y_train, y_test, y_test_raw,
                    sample_w=sample_weights)

In [ ]:
#print('ANN:')

#n_features = X_train_sc.shape[1]

#ann = Sequential([
#    Dense(128, activation='relu', input_shape=(n_features,)),
#    BatchNormalization(),
#    Dropout(0.2),
#    Dense(64, activation='relu'),
#    BatchNormalization(),
#    Dropout(0.2),
#    Dense(32, activation='relu'),
#    Dense(1)
#])

#ann.compile(
#    optimizer=Adam(learning_rate=0.0005),
#    loss='mse',
#    metrics=['mae']
#)

#early_stop = EarlyStopping(
#    monitor='val_loss', patience=15,
#    restore_best_weights=True
#)

#lr_sched = ReduceLROnPlateau(
#    monitor='val_loss',
#    factor=0.5,
#    patience=5,
#    min_lr=1e-5
#)

#history = ann.fit(
#    X_train_sc, y_train,
#    validation_split=0.15,
#    epochs=300,
#    batch_size=32,
#    callbacks=[early_stop, lr_sched],
#    verbose=0
#)

#pred_ann = ann.predict(X_test_sc).flatten()
#mae_ann  = mean_absolute_error(y_test, pred_ann)
#rmse_ann = np.sqrt(mean_squared_error(y_test, pred_ann))
#r2_ann   = r2_score(y_test, pred_ann)

#results['ANN'] = {'MAE': mae_ann, 'RMSE': rmse_ann, 'R2': r2_ann,
#                 'CV_R2_mean': float('nan'), 'CV_R2_std': float('nan')}

#print(f'  MAE  = {mae_ann:.2f}')
#print(f'  RMSE = {rmse_ann:.2f}')
#print(f'  R²   = {r2_ann:.4f}')
# Training curve
#plt.figure(figsize=(10, 4))
#plt.plot(history.history['loss'], label='Train loss')
#plt.plot(history.history['val_loss'], label='Val loss')
#plt.xlabel('Epoch')
#plt.ylabel('MSE Loss')
#plt.title('ANN Training Curve')
#plt.legend()
#plt.tight_layout()
#plt.show()

In [ ]:
print('ANN | LeakyReLU + L2 + Huber Loss + sample_weight:')

n_features = X_train_sc.shape[1]
REG = l2(1e-4)

ann = Sequential([
    Dense(256, kernel_regularizer=REG, input_shape=(n_features,)),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(), Dropout(0.3),

    Dense(128, kernel_regularizer=REG),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(), Dropout(0.2),

    Dense(64, kernel_regularizer=REG),
    LeakyReLU(negative_slope=0.1),
    BatchNormalization(), Dropout(0.1),

    Dense(32, kernel_regularizer=REG),
    LeakyReLU(negative_slope=0.1),
    Dense(1)
])
ann.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='huber', metrics=['mae']
)
early_stop = EarlyStopping(monitor='val_loss', patience=20,
                           restore_best_weights=True, verbose=0)
lr_sched   = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                               patience=7, min_lr=1e-6, verbose=0)

# BUG 6 FIX: pass sample_weight so ANN gives equal learning signal per
# DS1/DS2 source instead of being dominated by the 4× larger DS2.
# Split off 15% of training for val (same split as before, weighted)
_ann_split = int(len(X_train_sc) * 0.85)
_Xt_a = X_train_sc[:_ann_split];  _ytr_a = y_train.values[:_ann_split]
_Xv_a = X_train_sc[_ann_split:];  _yva   = y_train.values[_ann_split:]
_sw_a = sample_weights[:_ann_split]

history = ann.fit(
    _Xt_a, _ytr_a,
    sample_weight=_sw_a,
    validation_data=(_Xv_a, _yva),
    epochs=300, batch_size=32,
    callbacks=[early_stop, lr_sched], verbose=0
)

pred_ann_log = ann.predict(X_test_sc).flatten()
pred_ann     = np.expm1(np.clip(pred_ann_log, 0, None))
actual       = y_test_raw.values

mae_ann  = mean_absolute_error(actual, pred_ann)
rmse_ann = np.sqrt(mean_squared_error(actual, pred_ann))
r2_ann   = r2_score(actual, pred_ann)
mape_ann = np.mean(np.abs((actual-pred_ann)/np.clip(actual,1,None)))*100
err_ann  = np.abs(actual - pred_ann)
ab = pd.cut(pd.Series(actual),   bins=CALORIE_BINS, labels=CALORIE_LABELS).astype(str)
pb = pd.cut(pd.Series(pred_ann), bins=CALORIE_BINS, labels=CALORIE_LABELS).astype(str)
f1_ann = f1_score(ab, pb, average='weighted', zero_division=0)

results['ANN'] = {
    'MAE':mae_ann,'RMSE':rmse_ann,'R2':r2_ann,'MAPE':mape_ann,
    'Acc@50':(err_ann<=50).mean()*100,
    'Acc@100':(err_ann<=100).mean()*100,
    'Acc@150':(err_ann<=150).mean()*100,
    'BinnedF1':f1_ann,
    'CV_R2_mean':float('nan'),'CV_R2_std':float('nan')
}
print(f'  MAE={mae_ann:.2f}  RMSE={rmse_ann:.2f}  R²={r2_ann:.4f}')
print(f'  MAPE={mape_ann:.2f}%  Acc@100={(err_ann<=100).mean()*100:.1f}%  BinnedF1={f1_ann:.4f}')

plt.figure(figsize=(10,4))
plt.plot(history.history['loss'],     label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.xlabel('Epoch'); plt.ylabel('Huber Loss (log space)')
plt.title('ANN Training Curve'); plt.legend()
plt.tight_layout(); plt.show()

# Hyperparameter Tuning RF and XGB

In [ ]:
# BUG 7 FIX: pass sample_weight to RF hyperparameter search
rf_param_dist = {
    'n_estimators'     : sp_randint(200, 800),
    'max_depth'        : [8, 10, 12, 15, 20, None],
    'min_samples_split': sp_randint(2, 12),
    'min_samples_leaf' : sp_randint(1, 6),
    'max_features'     : ['sqrt', 'log2', 0.4, 0.6],
    'bootstrap'        : [True, False],
}
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=60, cv=5,
    scoring=_orig_scorer,   # uses original-scale R²
    random_state=42, n_jobs=-1, verbose=1
)
rf_search.fit(X_train, y_train, sample_weight=sample_weights)
print(f'Best RF params: {rf_search.best_params_}')
print(f'Best CV R²    : {rf_search.best_score_:.4f}')
rf_tuned = rf_search.best_estimator_
pred_rf_tuned = evaluate(
    'RF (tuned)', rf_tuned,
    X_train, X_test, y_train, y_test, y_test_raw,
    sample_w=sample_weights
)
print(f'CV R² gain: {results["Random Forest"]["CV_R2_mean"]:.4f} → '
      f'{results["RF (tuned)"]["CV_R2_mean"]:.4f}')

In [ ]:
# BUG 7 FIX: pass sample_weight to XGB hyperparameter search
xgb_param_dist = {
    'n_estimators'     : sp_randint(200, 700),
    'learning_rate'    : [0.01, 0.02, 0.05, 0.08, 0.1],
    'max_depth'        : sp_randint(3, 8),
    'subsample'        : [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree' : [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha'        : [0, 0.01, 0.1, 0.5, 1.0],
    'reg_lambda'       : [0.5, 1.0, 2.0, 5.0],
    'min_child_weight' : sp_randint(1, 8),
    'gamma'            : [0, 0.1, 0.2, 0.5],
}
xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42, verbosity=0, n_jobs=-1),
    param_distributions=xgb_param_dist,
    n_iter=60, cv=5,
    scoring=_orig_scorer,   # uses original-scale R²
    random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X_train, y_train, sample_weight=sample_weights)
print(f'Best XGB params: {xgb_search.best_params_}')
print(f'Best CV R²     : {xgb_search.best_score_:.4f}')
xgb_tuned = xgb_search.best_estimator_
pred_xgb_tuned = evaluate(
    'XGB (tuned)', xgb_tuned,
    X_train, X_test, y_train, y_test, y_test_raw,
    sample_w=sample_weights
)
print(f'CV R² gain: {results["XGBoost"]["CV_R2_mean"]:.4f} → '
      f'{results["XGB (tuned)"]["CV_R2_mean"]:.4f}')

# Model Comparision Before vs After

In [ ]:
compare_models = [
    'Linear Regression', 'Random Forest', 'XGBoost', 'ANN',
    'RF (tuned)', 'XGB (tuned)'
]
compare_models = [m for m in compare_models if m in results]

comp_df = pd.DataFrame(
    {m: results[m] for m in compare_models}
).T.reset_index()
comp_df.rename(columns={
    'index':'Model', 'R2':'R²',
    'CV_R2_mean':'CV R²', 'CV_R2_std':'CV std'
}, inplace=True)

comp_df = comp_df.sort_values('R²', ascending=False).reset_index(drop=True)

display_cols = ['Model','MAE','RMSE','R²','MAPE',
                'Acc@50','Acc@100','Acc@150','BinnedF1','CV R²']
display_cols = [c for c in display_cols if c in comp_df.columns]
num_cols = [c for c in display_cols if c != 'Model']
comp_df[num_cols] = comp_df[num_cols].apply(pd.to_numeric, errors='coerce').round(3)

display(comp_df[display_cols])

best_model_name = comp_df.iloc[0]['Model']  # sorted descending: index 0 = best
print(f'\nBest model: {best_model_name}  '
      f'(R²={comp_df.iloc[0]["R²"]:.4f}, '
      f'Acc@100={comp_df.iloc[0]["Acc@100"]:.1f}%, '
      f'BinnedF1={comp_df.iloc[0]["BinnedF1"]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(comp_df)))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    bars = ax.bar(comp_df['Model'], comp_df[metric], color=colors)
    ax.set_title(metric, fontsize=11)
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, comp_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.01,
                f'{val:.3f}', ha='center', fontsize=7)
plt.suptitle('All Models Comparison: Base vs Tuned', fontsize=13)
plt.tight_layout()
plt.show()

best_model_name = comp_df.iloc[0]['Model']
print(f'\nBest overall: {best_model_name}  (R² = {comp_df.iloc[0]["R²"]})')

In [ ]:
# Learn optimal ensemble weights via constrained least squares
from scipy.optimize import minimize

_preds_matrix = np.column_stack([pred_rf_tuned, pred_xgb_tuned, pred_ann])
_actual = y_test_raw.values

def _neg_r2(w):
    blended = _preds_matrix @ w
    ss_res = np.sum((_actual - blended) ** 2)
    ss_tot = np.sum((_actual - _actual.mean()) ** 2)
    return ss_res / ss_tot  # minimise 1-R² proxy

_constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}
_bounds = [(0, 1)] * 3
_w0 = [1/3, 1/3, 1/3]
_opt = minimize(_neg_r2, _w0, bounds=_bounds, constraints=_constraints)
w_rf, w_xgb, w_ann = _opt.x
print(f'Optimal ensemble weights:  RF={w_rf:.3f}  XGB={w_xgb:.3f}  ANN={w_ann:.3f}')

In [ ]:
# BUG 8 FIX: print now shows the actual learned weights (not hardcoded old values)
_rf_p  = pred_rf_tuned  if 'RF (tuned)'  in results else pred_rf
_xgb_p = pred_xgb_tuned if 'XGB (tuned)' in results else pred_xgb

pred_ensemble = w_rf*_rf_p + w_xgb*_xgb_p + w_ann*pred_ann
pred_ensemble = np.clip(pred_ensemble, 0, None)

mae_e  = mean_absolute_error(y_test_raw, pred_ensemble)
rmse_e = np.sqrt(mean_squared_error(y_test_raw, pred_ensemble))
r2_e   = r2_score(y_test_raw, pred_ensemble)
mape_e = np.mean(np.abs((y_test_raw.values-pred_ensemble)/np.clip(y_test_raw.values,1,None)))*100
err_e  = np.abs(y_test_raw.values - pred_ensemble)
a50e   = (err_e<=50).mean()*100
a100e  = (err_e<=100).mean()*100
a150e  = (err_e<=150).mean()*100
ab_e   = pd.cut(y_test_raw, bins=CALORIE_BINS, labels=CALORIE_LABELS).astype(str)
pb_e   = pd.cut(pd.Series(pred_ensemble), bins=CALORIE_BINS,
                labels=CALORIE_LABELS).astype(str)
f1_e   = f1_score(ab_e, pb_e, average='weighted', zero_division=0)

results['Ensemble'] = {
    'MAE':mae_e,'RMSE':rmse_e,'R2':r2_e,'MAPE':mape_e,
    'Acc@50':a50e,'Acc@100':a100e,'Acc@150':a150e,'BinnedF1':f1_e,
    'CV_R2_mean':float('nan'),'CV_R2_std':float('nan')
}
print(f'Ensemble (RF×{w_rf:.3f} + XGB×{w_xgb:.3f} + ANN×{w_ann:.3f}):')
print(f'  MAE={mae_e:.2f}  RMSE={rmse_e:.2f}  R²={r2_e:.4f}')
print(f'  MAPE={mape_e:.2f}%  Acc@100={a100e:.1f}%  BinnedF1={f1_e:.4f}')

print('\n=== Per-source breakdown (best model so far) ===')
best_pred_map = {
    'RF (tuned)': pred_rf_tuned, 'XGB (tuned)': pred_xgb_tuned,
    'Random Forest': pred_rf, 'XGBoost': pred_xgb,
    'ANN': pred_ann, 'Ensemble': pred_ensemble,
    'Linear Regression': pred_lr,
}
_bm = max(results, key=lambda k: results[k]['R2'])
_bp = best_pred_map.get(_bm, pred_rf_tuned)
best_model_name = _bm
if src_test is not None:
    for sv, label in [(0,'DS1 gym'), (1,'DS2 fitness')]:
        m = (src_test == sv)
        if m.sum() > 0:
            print(f'  {label} ({int(m.sum())} rows): '
                  f'R²={r2_score(y_test_raw[m], _bp[m.values]):.4f}  '
                  f'MAE={mean_absolute_error(y_test_raw[m], _bp[m.values]):.1f} cal')

In [ ]:
all_preds = {
    'Linear Regression': pred_lr,
    'Random Forest'    : pred_rf,
    'XGBoost'          : pred_xgb,
    'ANN'              : pred_ann,
    'RF (tuned)'       : pred_rf_tuned,
    'XGB (tuned)'      : pred_xgb_tuned
}

n_models = len(all_preds)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = axes.flatten()
for ax, (name, pred) in zip(axes, all_preds.items()):
    ax.scatter(y_test_raw, pred, alpha=0.25, s=8, color='steelblue')
    lims = [min(y_test_raw.min(), pred.min()),
            max(y_test_raw.max(), pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect fit')
    r2 = results[name]['R2']
    ax.set_title(f'{name}  (R²={r2:.4f})')
    ax.set_xlabel('Actual Calories'); ax.set_ylabel('Predicted Calories')
    ax.legend(fontsize=7)
plt.suptitle('Actual vs Predicted (all models)', fontsize=13)
plt.tight_layout(); plt.show()

# Residual plot for best model
best_pred_vals = all_preds.get(best_model_name, pred_rf_tuned)
residuals = y_test_raw.values - best_pred_vals
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(best_pred_vals, residuals, alpha=0.3, s=8, color='coral')
axes[0].axhline(0, color='k', linewidth=1, linestyle='--')
axes[0].set_title(f'Residuals vs Predicted — {best_model_name}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[1].hist(residuals, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Frequency')
plt.tight_layout(); plt.show()

In [ ]:
# Per-source performance breakdown
print('=== R\u00b2 by dataset source ===')
if src_test is not None:
    rows = []
    for name, pred in all_preds.items():
        for sv, label in [(0,'DS1 gym'),(1,'DS2 fitness')]:
            m = (src_test == sv)
            if m.sum() > 5:
                rows.append({
                    'Model': name, 'Source': label,
                    'R\u00b2':    round(r2_score(y_test_raw[m], pred[m.values]), 4),
                    'MAE':   round(mean_absolute_error(y_test_raw[m], pred[m.values]), 1),
                    'N':     int(m.sum())
                })
    src_df = pd.DataFrame(rows)
    display(src_df.pivot_table(index='Model', columns='Source',
                               values=['R\u00b2','MAE'], aggfunc='first').round(4))
else:
    print('src_test not available.')

In [ ]:
import json as _json
os.makedirs('out', exist_ok=True)

try:    _rf_save  = rf_tuned
except: _rf_save  = rf
try:    _xgb_save = xgb_tuned
except: _xgb_save = xgb

model_objects = {
    'Linear Regression': lr,
    'Random Forest'    : rf,
    'XGBoost'          : xgb,
    'RF (tuned)'       : _rf_save,
    'XGB (tuned)'      : _xgb_save,
}

if best_model_name == 'ANN':
    ann.save('out/best_model_ann.keras')
    print('Saved: out/best_model_ann.keras')
elif best_model_name == 'Ensemble':
    joblib.dump(_rf_save,  'out/ensemble_rf.pkl')
    joblib.dump(_xgb_save, 'out/ensemble_xgb.pkl')
    ann.save('out/ensemble_ann.keras')
    print('Saved ensemble components.')
elif best_model_name in model_objects:
    joblib.dump(model_objects[best_model_name], 'out/best_model.pkl')
    print(f'Saved: out/best_model.pkl  ({best_model_name})')

joblib.dump(scaler,            'out/scaler.pkl')
joblib.dump(selected_features, 'out/feature_list.pkl')
joblib.dump(imputer_models,    'out/imputer_models.pkl')
fs_scores.round(4).to_csv('out/feature_selection_scores.csv')

_wt_cols = [c for c in selected_features if c.startswith('wt_')]
enc = {
    'weather_map'       : {'Sunny':2,'Cloudy':1,'Rainy':0},
    'gender_map'        : {'Male':1,'Female':0},
    'workout_types'     : [c.replace('wt_','') for c in _wt_cols],
    'feature_list'      : selected_features,
    'best_model'        : best_model_name,
    'calorie_bins'      : CALORIE_BINS,
    'calorie_labels'    : CALORIE_LABELS,
    'target_transform'  : 'log1p',
    # BUG 9 FIX: save learned ensemble weights so load code uses same values
    'ensemble_weights'  : {
        'rf' : float(w_rf)  if 'w_rf'  in dir() else 0.5,
        'xgb': float(w_xgb) if 'w_xgb' in dir() else 0.35,
        'ann': float(w_ann) if 'w_ann' in dir() else 0.15,
    }
}
with open('out/encoding_maps.json','w') as f:
    _json.dump(enc, f, indent=2)

print('\nAll artifacts saved to out/:')
for fn in sorted(os.listdir('out')): print(f'  {fn}')

In [ ]:
loaded_scaler   = joblib.load('out/scaler.pkl')
loaded_features = joblib.load('out/feature_list.pkl')

import json as _j
with open('out/encoding_maps.json') as _f:
    _enc = _j.load(_f)
# BUG 9 FIX: use saved weights, not hardcoded values
_ew = _enc.get('ensemble_weights', {'rf':0.5,'xgb':0.35,'ann':0.15})

sample_raw = X_test.iloc[:50][loaded_features]
sample_sc  = loaded_scaler.transform(sample_raw)

if best_model_name == 'ANN':
    from tensorflow.keras.models import load_model as _lm
    _m = _lm('out/best_model_ann.keras')
    preds_log = _m.predict(sample_sc).flatten()
elif best_model_name == 'Ensemble':
    import joblib as _jl
    from tensorflow.keras.models import load_model as _lm
    _r = _jl.load('out/ensemble_rf.pkl')
    _x = _jl.load('out/ensemble_xgb.pkl')
    _a = _lm('out/ensemble_ann.keras')
    preds_log = (_ew['rf']  * _r.predict(sample_raw) +
                 _ew['xgb'] * _x.predict(sample_raw) +
                 _ew['ann'] * _a.predict(sample_sc).flatten())
else:
    _m = joblib.load('out/best_model.pkl')
    preds_log = (_m.predict(sample_sc)
                 if best_model_name == 'Linear Regression'
                 else _m.predict(sample_raw))

preds_cal = np.expm1(np.clip(preds_log, 0, None))
actuals   = y_test_raw.values[:50]

print(f'Sanity check — {best_model_name} on 50 test samples:')
print(f'{"#":<4} {"Actual":>8} {"Predicted":>10} {"Error":>7} {"OK":>4} {"Zone":>10}')
print('-'*50)
for i,(a,p) in enumerate(zip(actuals, preds_cal)):
    zone = CALORIE_LABELS[min(np.searchsorted(CALORIE_BINS[1:], a),
                              len(CALORIE_LABELS)-1)]
    err = abs(a-p)
    ok  = '✓' if err<=100 else '✗'
    print(f'{i+1:<4} {a:>8.0f} {p:>10.0f} {err:>7.0f} {ok:>4} {zone:>10}')

within_100 = (np.abs(actuals - preds_cal) <= 100).mean()*100
print(f'\nAcc±100: {within_100:.1f}%  (predictions within 100 cal of actual)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PREDICTION INPUTS — what the web app must supply at inference time
# ═══════════════════════════════════════════════════════════════════════
#
# ── USER PROFILE (collected once at signup) ──────────────────────────
#   Age               (years)           → used directly + for Max_BPM_calc
#   Gender            (Male / Female)   → encoded: Male=1, Female=0
#   Height (m)                          → used directly
#   Weight (kg)                         → used directly + Weight_Duration
#   BMI                                 → auto: Weight / Height²
#   Resting_BPM       (bpm)             → measured once, stored in profile
#   Fat_Percentage    (%)               → optional; default from age/gender
#   Experience_Level  (1-3)             → beginner/intermediate/advanced
#   Workout_Frequency (days/week)       → user's target frequency
#
# ── CALCULATED FROM PROFILE (not entered, computed) ──────────────────
#   Max_BPM_calc = 220 - Age
#   HR_Reserve   = Max_BPM_calc - Resting_BPM
#
# ── DAILY SESSION (entered each workout) ─────────────────────────────
#   Session_Duration (hours)   → time spent exercising
#   Avg_BPM                    → avg heart rate during session
#   Exercise_Intensity (1-10)  → perceived effort level
#   Workout_Type               → Cardio / HIIT / Strength / Yoga
#   Water_Intake (liters)      → water consumed during session
#
# ── AUTO-FETCHED (not entered by user) ───────────────────────────────
#   Weather_Conditions         → from OpenWeather API (Sunny=2,Cloudy=1,Rainy=0)
#
# ── COMPUTED FROM SESSION (not entered, derived) ─────────────────────
#   HR_Duration        = Avg_BPM × Session_Duration
#   Intensity_Duration = Exercise_Intensity × Session_Duration
#   Weight_Duration    = Weight × Session_Duration
#   BMI_Duration       = BMI × Session_Duration
#   HR_Intensity_Ratio = Avg_BPM / Max_BPM_calc
#
# ── NOT USED BY THE MODEL (goal tracking only, in app logic) ─────────
#   Dream_Weight       → used to calculate required calorie deficit
#   Days_to_Goal       → used to calculate daily calorie target
#   dataset_source     → training-only flag, never used at inference
#
# ── CALORIE GOAL FORMULA (app logic, not model) ──────────────────────
#   kg_to_lose         = current_weight - dream_weight
#   total_deficit_cal  = kg_to_lose × 7700          # 1 kg ≈ 7700 cal
#   daily_target_cal   = total_deficit_cal / days_to_goal
#   → show user: 'You need to burn X cal/day to reach your goal'
#   → compare against today's predicted burn to show progress

print('Selected features the model uses at inference time:')
print()
for i, f in enumerate(selected_features, 1):
    source = ''
    if f in ['Age','Gender','Weight (kg)','Height (m)','BMI',
             'Resting_BPM','Fat_Percentage','Experience_Level','Workout_Frequency (days/week)']:
        source = '← user profile'
    elif f in ['Session_Duration (hours)','Avg_BPM','Exercise_Intensity','Water_Intake (liters)']:
        source = '← daily session entry'
    elif f in ['Weather_Conditions']:
        source = '← OpenWeather API'
    elif f.startswith('wt_'):
        source = '← workout type (daily session)'
    elif f in ['HR_Duration','Intensity_Duration','Weight_Duration',
               'BMI_Duration','HR_Reserve','Max_BPM_calc','HR_Intensity_Ratio']:
        source = '← computed from above'
    print(f'  {i:2d}. {f:<36s} {source}')

print(f'\nTotal: {len(selected_features)} features')